In [2]:
!pip install optuna mlflow xgboost imbalanced-learn openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

In [4]:
from xgboost import XGBClassifier

In [5]:
from google.colab import files
uploaded = files.upload()

Saving train_transaction.xls to train_transaction.xls


In [6]:
df = pd.read_excel("train_transaction.xls")

In [12]:
import pandas as pd
import numpy as np


print("=== DATASET SHAPE ===")
print(df.shape)


print("\n=== FIRST 5 ROWS ===")
print(df.head())


print("\n=== DATASET INFO ===")
df.info()


print("\n=== TARGET DISTRIBUTION (COUNT) ===")
print(df["isFraud"].value_counts())

print("\n=== TARGET DISTRIBUTION (%) ===")
print(df["isFraud"].value_counts(normalize=True) * 100)


print("\n=== TOP 20 MISSING VALUES ===")
missing = df.isnull().sum().sort_values(ascending=False)
print(missing.head(20))


print("\n=== DUPLICATE ROWS ===")
print(df.duplicated().sum())


print("\n=== DATA TYPES SUMMARY ===")
print(df.dtypes.value_counts())

=== DATASET SHAPE ===
(65535, 256)

=== FIRST 5 ROWS ===
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V192  V193  V194  V195  V196 V197  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
3  567.0  150.0  mastercard  117.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
4  514.0  150.0  mastercard  102.0  ...  1.0   1.0   1.0   1.0   1.0  1.0   

  V198  V199  V20

In [13]:
from sklearn.model_selection import train_test_split


missing_percent = df.isnull().mean() * 100
drop_cols = missing_percent[missing_percent > 70].index.tolist()

print("Columns to drop:", len(drop_cols))

df_clean = df.drop(columns=drop_cols)


if "TransactionID" in df_clean.columns:
    df_clean = df_clean.drop(columns=["TransactionID"])


X = df_clean.drop("isFraud", axis=1)
y = df_clean["isFraud"]


categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", len(categorical_cols))
print("Numerical columns:", len(numerical_cols))


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Columns to drop: 41
Categorical columns: 10
Numerical columns: 203
Train shape: (52428, 213)
Test shape: (13107, 213)


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

print("Preprocessing pipeline ready.")

Preprocessing pipeline ready.


In [15]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

print("SMOTE ready.")

SMOTE ready.


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

log_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", smote),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)
y_prob_log = log_model.predict_proba(X_test)[:, 1]

print("=== LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_log))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_log))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

=== LOGISTIC REGRESSION ===
              precision    recall  f1-score   support

           0       0.99      0.81      0.89     12760
           1       0.09      0.74      0.17       347

    accuracy                           0.80     13107
   macro avg       0.54      0.77      0.53     13107
weighted avg       0.97      0.80      0.87     13107

ROC-AUC: 0.8538394478422303
Confusion Matrix:
[[10294  2466]
 [   91   256]]


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", smote),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print("=== RANDOM FOREST ===")
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

=== RANDOM FOREST ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     12760
           1       0.91      0.47      0.62       347

    accuracy                           0.98     13107
   macro avg       0.95      0.74      0.81     13107
weighted avg       0.98      0.98      0.98     13107

ROC-AUC: 0.9303625793862302
Confusion Matrix:
[[12743    17]
 [  183   164]]


In [20]:
!pip install xgboost

In [21]:
from xgboost import XGBClassifier

xgb_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", smote),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost done")

XGBoost done


In [24]:
from xgboost import XGBClassifier

xgb_model = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", smote),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBOOST ===")
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

=== XGBOOST ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     12760
           1       0.63      0.52      0.57       347

    accuracy                           0.98     13107
   macro avg       0.81      0.76      0.78     13107
weighted avg       0.98      0.98      0.98     13107

ROC-AUC: 0.9209640627681968
Confusion Matrix:
[[12654   106]
 [  166   181]]


In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Precision": [
        precision_score(y_test, y_pred_log),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_xgb)
    ],
    "Recall": [
        recall_score(y_test, y_pred_log),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_xgb)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_log),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_xgb)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_log),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb)
    ]
})

print(results)

                 Model  Precision    Recall  F1 Score   ROC-AUC
0  Logistic Regression   0.094048  0.737752  0.166830  0.853839
1        Random Forest   0.906077  0.472622  0.621212  0.930363
2              XGBoost   0.630662  0.521614  0.570978  0.920964


In [26]:
!pip install optuna mlflow

In [27]:
# =========================
# OPTUNA TUNING - RANDOM FOREST
# =========================

import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from imblearn.pipeline import Pipeline as ImbPipeline

def objective(trial):

    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 5, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)

    model = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", smote),
        ("model", RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    return f1_score(y_test, preds)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print("Best Parameters:")
print(study.best_params)

print("Best F1 Score:")
print(study.best_value)

[I 2026-05-16 13:27:54,909] A new study created in memory with name: no-name-d526108d-9e47-4e8e-b49b-0ccb2f5ced5c
[I 2026-05-16 13:28:27,586] Trial 0 finished with value: 0.35148514851485146 and parameters: {'n_estimators': 99, 'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 1}. Best is trial 0 with value: 0.35148514851485146.
[I 2026-05-16 13:28:45,582] Trial 1 finished with value: 0.29568788501026694 and parameters: {'n_estimators': 79, 'max_depth': 7, 'min_samples_split': 5, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.35148514851485146.
[I 2026-05-16 13:29:40,789] Trial 2 finished with value: 0.5955414012738853 and parameters: {'n_estimators': 163, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 1}. Best is trial 2 with value: 0.5955414012738853.
[I 2026-05-16 13:29:52,657] Trial 3 finished with value: 0.2340316811446091 and parameters: {'n_estimators': 61, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 2}. Best is trial 2 with value: 

Best Parameters:
{'n_estimators': 163, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 1}
Best F1 Score:
0.5955414012738853


In [28]:
best_params = study.best_params

In [29]:
best_rf = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", smote),
    ("model", RandomForestClassifier(
        **best_params,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

best_rf.fit(X_train, y_train)

best_pred = best_rf.predict(X_test)
best_prob = best_rf.predict_proba(X_test)[:, 1]

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(classification_report(y_test, best_pred))
print("ROC-AUC:", roc_auc_score(y_test, best_prob))
print(confusion_matrix(y_test, best_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99     12760
           1       0.67      0.54      0.60       347

    accuracy                           0.98     13107
   macro avg       0.83      0.77      0.79     13107
weighted avg       0.98      0.98      0.98     13107

ROC-AUC: 0.9226265436838823
[[12666    94]
 [  160   187]]


In [30]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("Fraud Detection ML")

with mlflow.start_run():

    mlflow.log_params(best_params)

    mlflow.log_metric("f1_score", f1_score(y_test, best_pred))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, best_prob))

    mlflow.sklearn.log_model(best_rf, "random_forest_model")

print("MLflow tracking completed.")

2026/05/16 13:35:39 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/16 13:35:39 INFO mlflow.store.db.utils: Updating database tables
2026/05/16 13:35:42 INFO mlflow.tracking.fluent: Experiment with name 'Fraud Detection ML' does not exist. Creating a new experiment.
2026/05/16 13:35:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/16 13:35:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow tracking completed.


In [ ]:
!mlflow ui

Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/05/16 13:36:45 INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
2026/05/16 13:36:45 INFO:     Started parent process [22965]
2026/05/16 13:37:06 INFO:     Started server process [22971]
2026/05/16 13:37:06 INFO:     Waiting for application startup.
2026/05/16 13:37:06 INFO:     Application startup complete.
2026/05/16 13:37:07 INFO:     Started server process [22968]
2026/05/16 13:37:07 INFO:     Waiting for application startup.
2026/05/16 13:37:07 INFO:     Application startup complete.
2026/05/16 13:37:07 INFO:     Started server process [22969]
2026/05/16 13:37:07 INFO:     Waiting for application startup.
2026/05/16 13:37:07 INFO:     Application st